# Topic 19 — Dimensionality Reduction
### Theory → PCA from scratch (using eigenvectors from Topic 4) → sklearn PCA → t-SNE → UMAP.

**High-dimensional data** (e.g. TF-IDF text vectors with thousands of columns, Topic 24) is hard to
visualize and can slow down or confuse some algorithms. Dimensionality reduction compresses many
features into fewer, while trying to **preserve variance** (the information that actually
distinguishes data points).

In [ ]:
!pip install umap-learn -q
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
import umap

rng = np.random.default_rng(0)

## 1. PCA — the core idea, connected to Topic 4's eigenvectors

PCA finds new axes (**principal components**) that are:
1. Linear combinations of the original features,
2. Ordered by how much variance they capture (PC1 captures the most, PC2 the second most, etc.),
3. Mathually, the eigenvectors of the data's **covariance matrix** — this is exactly the
   eigenvalue/eigenvector concept from Topic 4, applied to real data.

In [ ]:
# Correlated 2D data -- most of the "spread" is along one diagonal direction
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]   # strong positive correlation
X = rng.multivariate_normal(mean, cov, size=200)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6)
plt.axis("equal")
plt.title("Correlated 2D data")
plt.show()

In [ ]:
# PCA from scratch, using eigenvectors of the covariance matrix (Topic 4)
X_centered = X - X.mean(axis=0)             # PCA requires mean-centered data
cov_matrix = np.cov(X_centered.T)
eigvals, eigvecs = np.linalg.eig(cov_matrix)

# Sort by eigenvalue, descending -- largest eigenvalue = direction of most variance
sort_idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[sort_idx], eigvecs[:, sort_idx]

print("eigenvalues (variance captured by each component):", eigvals)
print("eigenvectors (principal component directions):\n", eigvecs)

plt.figure(figsize=(5, 5))
plt.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.4)
origin = np.zeros(2)
for i in range(2):
    vec = eigvecs[:, i] * np.sqrt(eigvals[i]) * 2   # scaled for visibility
    plt.quiver(*origin, *vec, angles="xy", scale_units="xy", scale=1,
               color=["red", "blue"][i], label=f"PC{i+1}")
plt.axis("equal")
plt.legend()
plt.title("Principal components = directions of max variance")
plt.show()
# PC1 (red) points along the direction the data is MOST spread out. PC2 (blue) is perpendicular,
# capturing whatever variance is left.

In [ ]:
# Project the data onto just PC1 -- this IS dimensionality reduction (2D -> 1D)
X_pc1_only = X_centered @ eigvecs[:, 0]
print("original shape:", X_centered.shape, " reduced shape:", X_pc1_only.shape)
print("variance explained by PC1 alone:", eigvals[0] / eigvals.sum())

## 2. sklearn's PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print("explained variance ratio per component:", pca.explained_variance_ratio_)
# Should closely match your from-scratch eigenvalue proportions above.

plt.figure(figsize=(5, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Data in PCA-transformed space (axes are now uncorrelated)")
plt.axis("equal")
plt.show()

## 3. PCA on real higher-dimensional data (digits dataset)

Each image is a 64-dimensional vector (8x8 pixels). PCA compresses this down to 2D so we can plot it,
while keeping as much distinguishing information as possible.

In [ ]:
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
print("original shape:", X_digits.shape)   # 1797 samples, 64 features each

pca_digits = PCA(n_components=2)
X_digits_pca = pca_digits.fit_transform(X_digits)
print("reduced shape:", X_digits_pca.shape)
print("variance explained by first 2 components:", pca_digits.explained_variance_ratio_.sum())

plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], c=y_digits, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, label="digit")
plt.title("PCA: 64 pixel features compressed to 2D")
plt.show()
# Notice: some digit clusters (e.g. 0 vs 6) separate reasonably well even in just 2D,
# but others overlap heavily -- PCA keeps only ~30% of the variance in 2 dimensions here,
# so some information is necessarily lost.

## 4. Choosing how many components to keep

Plot cumulative explained variance vs number of components, and pick enough to retain e.g. 90-95%.

In [ ]:
pca_full = PCA().fit(X_digits)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(6, 4))
plt.plot(cumulative_var)
plt.axhline(0.9, color="red", linestyle="--", label="90% variance")
plt.xlabel("number of components")
plt.ylabel("cumulative explained variance")
plt.legend()
plt.title("How many components do we need?")
plt.show()

n_for_90pct = np.argmax(cumulative_var >= 0.9) + 1
print(f"components needed for 90% variance: {n_for_90pct} (out of 64 original features)")

## 5. t-SNE — for visualization, not general-purpose reduction

Unlike PCA (which preserves global variance/structure), **t-SNE** focuses on preserving LOCAL
neighborhoods — points close together in high-D stay close in the 2D plot. Excellent for visual
exploration, but: non-linear, has no simple "explained variance" concept, is slower, and different
runs/perplexity settings can give visually different layouts (distances between distant clusters
aren't meaningfully interpretable).

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_digits_tsne = tsne.fit_transform(X_digits)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], c=y_digits, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, label="digit")
plt.title("t-SNE: same data, much cleaner cluster separation")
plt.show()
# t-SNE usually separates classes MUCH more visibly than PCA -- because it's specifically
# optimizing for that, not for preserving overall variance/distances.

## 6. UMAP — similar goal to t-SNE, usually faster, better global structure

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
X_digits_umap = reducer.fit_transform(X_digits)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_digits_umap[:, 0], X_digits_umap[:, 1], c=y_digits, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, label="digit")
plt.title("UMAP")
plt.show()

print("Rule of thumb:")
print("- PCA: fast, linear, interpretable, good default / preprocessing step before modeling")
print("- t-SNE / UMAP: best for VISUALIZATION and exploring cluster structure, not as model inputs")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change n_components in the from-scratch example to keep only PC1, reconstruct approximate
#    2D points from it (X_pc1_only @ eigvecs[:,0].reshape(1,-1)) and compare to the original X visually.
# 2. Re-run TSNE with perplexity=5 and perplexity=100 -- how different does the layout look?
# 3. Try PCA with n_components=0.95 (a float means "keep enough components for 95% variance") directly.
# 4. Once you have your real cyberbullying TF-IDF features (Topic 24), which of PCA/t-SNE/UMAP would
#    you reach for to visually check whether bullying vs non-bullying comments separate at all?
#    Why?

---
### Next up: **Topic 20 — Imbalanced Datasets** (high priority for your cyberbullying research).

Say "next" when you're ready.